In [1]:
import altair as alt
import duckdb
import pandas as pd

In [2]:
alt.renderers.enable('png')

RendererRegistry.enable('png')

In [2]:
conn = duckdb.connect()

In [3]:
conn.execute("create table if not exists weather_data as select * from 'nyc_weather_data.csv'")
conn.execute("attach 'citibike_data.duckdb' as cb_data;")
conn.execute("set memory_limit = '4GB';")

In [4]:
conn.sql('describe cb_data.rides')

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ ride_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rideable_type      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ started_at         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ ended_at           │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_lat          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │

In [5]:
conn.sql('describe weather_data')

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                 │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_max_f           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_min_f           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_mean_f          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ feels_like_max_f     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ feels_like_min_f     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ precipitation_in     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ rain_in              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ snowfall_in          │ DOUBLE      │ YES     │ NUL

In [6]:
df = conn.execute("""
    from cb_data.rides
    select date_trunc('month',ended_at) as month ,count(1) num_rides
    group by month
""").df()

In [7]:
alt.Chart(df).mark_bar(size=15).encode(
    alt.X('month'),
    alt.Y('num_rides'),
    tooltip=['month', 'num_rides']
).interactive()

ValueError: Saving charts in 'png' format requires the vl-convert-python package: see https://altair-viz.github.io/user_guide/saving_charts.html#png-svg-and-pdf-format

alt.Chart(...)

In [ ]:
alt.Chart(
    conn.execute("""
        with citibike_grouped_by_day as (
            select date_trunc('day',ended_at) as date,count(1) num_rides
                from cb_data.rides
            group by 1
        )
        select temp_max_f, num_rides, date
        from weather_data join citibike_grouped_by_day using(date)
        order by date asc
    """).df()
).mark_point().encode(
    x='temp_max_f',
    y='num_rides',
    tooltip=['temp_max_f', 'num_rides']
)

In [ ]:
alt.Chart(
    conn.execute(
        """
        select extract('hour' from started_at) sa, count(1) num_rides
        from cb_data.rides
        group by sa
        order by sa asc
        """
    ).df()
).mark_line().encode(x='sa',y='num_rides').interactive()

In [ ]:
# SELECT dayname(DATE '2026-03-11');
base = alt.Chart(
    conn.execute(
        """
        select extract('dow' FROM started_at) dow, count(1) num_rides
        from cb_data.rides
        group by dow
        order by dow asc
        """
    ).df()
).encode(x='dow:N',y=alt.Y('num_rides:Q'),tooltip=['dow','num_rides'])

base.mark_point().interactive() + base.mark_line()

#     y=alt.Y('num_rides:Q', scale=alt.Scale(domain=[4000, 8000], zero=False))

In [ ]:
BOROUGHS_URL = (
    "https://raw.githubusercontent.com/codeforgermany/click_that_hood/"
    "main/public/data/new-york-city-boroughs.geojson"
)

points_df = conn.execute(
    """
    select
        start_station_name as name,
        any_value(start_lat) as lat,
        any_value(start_lng) as lon
    from 
        cb_data.rides
    where name is not null
    group by name
    """
).df()

# ── Layer 1: Borough shapes (background map) ───────────────────────────────
geo_data = alt.Data(
    url=BOROUGHS_URL,
    format=alt.DataFormat(property="features", type="json"),
)

base_map = (
    alt.Chart(geo_data)
    .mark_geoshape(fill="lightgrey", stroke="#666", strokeWidth=1.5)
)

# ── Layer 2: Point markers ─────────────────────────────────────────────────
dots = (
    alt.Chart(points_df)
    .mark_circle(size=20, opacity=0.7, color="black")
    .encode(
        longitude="lon:Q",
        latitude="lat:Q",
        tooltip=[alt.Tooltip("name:N", title="Station")],
    )
)
zoom = alt.selection_interval(bind="scales")

# ── Compose & configure ────────────────────────────────────────────────────
chart = (
    (base_map + dots)
    .add_params(zoom)
    .project(
        type="mercator",
        scale=55000,
        center=[-73.95, 40.73],
        translate=[350, 350],
    )
    .properties(
        width=700,
        height=700,
    )
    .configure_view(strokeWidth=0)
)

chart

In [ ]:
import folium

m = folium.Map(location=[40.73, -73.95], zoom_start=12, tiles="CartoDB positron")

for _, row in points_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="black",
        fill=True,
        fill_opacity=0.7,
        tooltip=row["name"],
    ).add_to(m)

m

In [11]:
# alt.data_transformers.disable_max_rows()

# alt.Chart(conn.execute("""
#     select start_station_name,end_station_name,datepart('minute',ended_at-started_at) as duration
#     from cb_data.rides
#     using sample 5000;
# """).df()).transform_window(
#     index='count()'
# ).transform_fold(
#     ['Start Station Name', 'End Station Name','Duration (minutes)']
# ).mark_line().encode(
#     x='start_station_name:O',
#     y='end_station_name:O',
#     # color='Species:N',
#     # detail='index:N',
#     # opacity=alt.value(0.5)
# )

In [4]:
import json
from pathlib import Path

alt.renderers.enable("default")
alt.data_transformers.disable_max_rows()

NYC_BOUNDS = {
    "min_lat": 40.55,
    "max_lat": 40.95,
    "min_lon": -74.10,
    "max_lon": -73.65,
}


def point_in_ring(lon, lat, ring):
    inside = False
    for idx in range(len(ring)):
        x1, y1 = ring[idx]
        x2, y2 = ring[(idx + 1) % len(ring)]
        intersects = ((y1 > lat) != (y2 > lat)) and (
            lon < (x2 - x1) * (lat - y1) / ((y2 - y1) or 1e-12) + x1
        )
        if intersects:
            inside = not inside
    return inside


def point_in_polygon(lon, lat, polygon):
    min_lon, min_lat, max_lon, max_lat = polygon["bbox"]
    if lon < min_lon or lon > max_lon or lat < min_lat or lat > max_lat:
        return False
    if not point_in_ring(lon, lat, polygon["outer"]):
        return False
    for hole in polygon["holes"]:
        if point_in_ring(lon, lat, hole):
            return False
    return True


def load_neighborhoods(path):
    with Path(path).open() as handle:
        geojson = json.load(handle)

    neighborhoods = []
    for feature in geojson["features"]:
        geometry = feature["geometry"]
        polygons = []
        for polygon_coords in geometry["coordinates"]:
            outer = polygon_coords[0]
            holes = polygon_coords[1:]
            lons = [coord[0] for coord in outer]
            lats = [coord[1] for coord in outer]
            polygons.append(
                {
                    "outer": outer,
                    "holes": holes,
                    "bbox": (min(lons), min(lats), max(lons), max(lats)),
                }
            )

        feature_lons = [poly["bbox"][0] for poly in polygons] + [poly["bbox"][2] for poly in polygons]
        feature_lats = [poly["bbox"][1] for poly in polygons] + [poly["bbox"][3] for poly in polygons]
        neighborhoods.append(
            {
                "neighborhood": feature["properties"]["ntaname"],
                "borough": feature["properties"]["boroname"],
                "label": f"{feature['properties']['ntaname']} ({feature['properties']['boroname']})",
                "polygons": polygons,
                "bbox": (
                    min(feature_lons),
                    min(feature_lats),
                    max(feature_lons),
                    max(feature_lats),
                ),
            }
        )
    return neighborhoods


def lookup_neighborhood(lon, lat, neighborhoods):
    for neighborhood in neighborhoods:
        min_lon, min_lat, max_lon, max_lat = neighborhood["bbox"]
        if lon < min_lon or lon > max_lon or lat < min_lat or lat > max_lat:
            continue
        for polygon in neighborhood["polygons"]:
            if point_in_polygon(lon, lat, polygon):
                return neighborhood["neighborhood"], neighborhood["label"]
    return None, None


def load_station_lookup(column_prefix):
    return conn.execute(
        f"""
        select
            {column_prefix}_station_name as station_name,
            any_value({column_prefix}_lat) as lat,
            any_value({column_prefix}_lng) as lon
        from cb_data.rides
        where {column_prefix}_station_name is not null
          and {column_prefix}_lat between {NYC_BOUNDS['min_lat']} and {NYC_BOUNDS['max_lat']}
          and {column_prefix}_lng between {NYC_BOUNDS['min_lon']} and {NYC_BOUNDS['max_lon']}
        group by 1
        """
    ).df()


def assign_neighborhoods(stations, neighborhoods):
    mapped_rows = []
    for station in stations.itertuples(index=False):
        neighborhood, label = lookup_neighborhood(station.lon, station.lat, neighborhoods)
        mapped_rows.append(
            {
                "station_name": station.station_name,
                "neighborhood": neighborhood,
                "neighborhood_label": label,
            }
        )
    return pd.DataFrame(mapped_rows)


neighborhoods = load_neighborhoods("nta_2020.geojson")
start_station_neighborhoods = assign_neighborhoods(load_station_lookup("start"), neighborhoods)
end_station_neighborhoods = assign_neighborhoods(load_station_lookup("end"), neighborhoods)

conn.register("start_station_neighborhoods", start_station_neighborhoods)
conn.register("end_station_neighborhoods", end_station_neighborhoods)

neighborhood_flow_df = conn.execute(
    """
    with departures as (
        select
            extract('hour' from started_at) as hour_of_day,
            case
                when extract('isodow' from started_at) in (6, 7) then 'Weekend'
                else 'Weekday'
            end as day_type,
            member_casual as rider_type,
            m.neighborhood,
            m.neighborhood_label,
            count(*) as departures
        from cb_data.rides
        join start_station_neighborhoods m
          on cb_data.rides.start_station_name = m.station_name
        where m.neighborhood is not null
        group by 1, 2, 3, 4, 5
    ),
    arrivals as (
        select
            extract('hour' from ended_at) as hour_of_day,
            case
                when extract('isodow' from ended_at) in (6, 7) then 'Weekend'
                else 'Weekday'
            end as day_type,
            member_casual as rider_type,
            m.neighborhood,
            m.neighborhood_label,
            count(*) as arrivals
        from cb_data.rides
        join end_station_neighborhoods m
          on cb_data.rides.end_station_name = m.station_name
        where m.neighborhood is not null
        group by 1, 2, 3, 4, 5
    ),
    neighborhood_hour as (
        select
            coalesce(a.hour_of_day, d.hour_of_day) as hour_of_day,
            coalesce(a.day_type, d.day_type) as day_type,
            coalesce(a.rider_type, d.rider_type) as rider_type,
            coalesce(a.neighborhood, d.neighborhood) as neighborhood,
            coalesce(a.neighborhood_label, d.neighborhood_label) as neighborhood_label,
            coalesce(arrivals, 0) as arrivals,
            coalesce(departures, 0) as departures,
            coalesce(arrivals, 0) - coalesce(departures, 0) as net_flow,
            coalesce(arrivals, 0) + coalesce(departures, 0) as total_activity
        from arrivals a
        full outer join departures d
          on a.hour_of_day = d.hour_of_day
         and a.day_type = d.day_type
         and a.rider_type = d.rider_type
         and a.neighborhood = d.neighborhood
    ),
    top_neighborhoods as (
        select
            neighborhood,
            neighborhood_label,
            sum(total_activity) as neighborhood_activity
        from neighborhood_hour
        group by 1, 2
        order by neighborhood_activity desc
        limit 80
    )
    select
        n.hour_of_day,
        n.day_type,
        n.rider_type,
        n.neighborhood,
        n.neighborhood_label,
        n.arrivals,
        n.departures,
        n.net_flow,
        n.total_activity,
        t.neighborhood_activity
    from neighborhood_hour n
    join top_neighborhoods t using (neighborhood, neighborhood_label)
    order by t.neighborhood_activity desc, n.neighborhood_label asc, n.hour_of_day asc
    """
).df()

neighborhood_sort = (
    neighborhood_flow_df[["neighborhood_label", "neighborhood_activity"]]
    .drop_duplicates()
    .sort_values(["neighborhood_activity", "neighborhood_label"], ascending=[False, True])["neighborhood_label"]
    .tolist()
)

max_abs_net_flow = float(neighborhood_flow_df["net_flow"].abs().quantile(0.98))
if max_abs_net_flow <= 0:
    max_abs_net_flow = float(neighborhood_flow_df["net_flow"].abs().max() or 1.0)

rider_param = alt.param(
    name="rider_type",
    value="member",
    bind=alt.binding_select(options=["member", "casual"], name="Rider type "),
)
day_param = alt.param(
    name="day_type",
    value="Weekday",
    bind=alt.binding_select(options=["Weekday", "Weekend"], name="Day type "),
)

alt.Chart(neighborhood_flow_df).add_params(
    rider_param,
    day_param,
).transform_filter(
    "datum.rider_type == rider_type"
).transform_filter(
    "datum.day_type == day_type"
).mark_rect().encode(
    x=alt.X("hour_of_day:O", title="Hour of day"),
    y=alt.Y("neighborhood_label:N", sort=neighborhood_sort, title="Neighborhood"),
    color=alt.Color(
        "net_flow:Q",
        title="Net flow",
        scale=alt.Scale(
            domain=[-max_abs_net_flow, 0, max_abs_net_flow],
            range=["#4c78a8", "#f7f7f7", "#e45756"],
            clamp=True,
        ),
    ),
    tooltip=[
        alt.Tooltip("neighborhood_label:N", title="Neighborhood"),
        alt.Tooltip("hour_of_day:O", title="Hour"),
        alt.Tooltip("day_type:N", title="Day type"),
        alt.Tooltip("rider_type:N", title="Rider type"),
        alt.Tooltip("arrivals:Q", format=",", title="Arrivals"),
        alt.Tooltip("departures:Q", format=",", title="Departures"),
        alt.Tooltip("net_flow:Q", format=",", title="Net flow"),
        alt.Tooltip("total_activity:Q", format=",", title="Hourly activity"),
        alt.Tooltip("neighborhood_activity:Q", format=",", title="Neighborhood activity"),
    ],
).properties(
    title="Citi Bike Sink/Source Heatmap by NYC Neighborhood",
    width=700,
    height=max(900, len(neighborhood_sort) * 14),
)


alt.Chart(...)